# 01 - Data Exploration

**QZero**: Quantum Zero-Day Detection System

This notebook explores the UGRansome dataset and synthetic data for zero-day attack detection.

## Contents
1. Dataset Overview
2. Synthetic Data Generation
3. Feature Analysis
4. Class Distribution
5. Correlation Analysis
6. Dimensionality Reduction (PCA/t-SNE)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from src.preprocessing.data_loader import (
    generate_synthetic_data, 
    DataLoader,
    UGRANSOME_FEATURES,
    ATTACK_CLASSES
)

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

## 1. Dataset Overview

UGRansome dataset features (14 network traffic attributes):

In [ ]:
print("UGRansome Features:")
for i, feature in enumerate(UGRANSOME_FEATURES, 1):
    print(f"  {i:2d}. {feature}")

print("\nAttack Classes:")
for name, code in ATTACK_CLASSES.items():
    print(f"  {code}: {name}")

## 2. Synthetic Data Generation

Generate synthetic data that mimics UGRansome patterns for development:

In [ ]:
# Generate synthetic data
X, y = generate_synthetic_data(
    n_samples=10000,
    n_features=14,
    anomaly_ratio=0.15,
    signature_ratio=0.20,
    random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"\nClass distribution:")
class_names = {0: 'Normal', 1: 'Anomaly (Zero-Day)', 2: 'Signature (Known)'}
for cls, name in class_names.items():
    count = (y == cls).sum()
    pct = count / len(y) * 100
    print(f"  {name}: {count:,} ({pct:.1f}%)")

## 3. Feature Analysis

In [ ]:
# Basic statistics
print("Feature Statistics:")
X.describe().round(2)

In [ ]:
# Feature distributions by class
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

df = X.copy()
df['class'] = y.map(class_names)

for i, feature in enumerate(UGRANSOME_FEATURES):
    ax = axes[i]
    for cls_code, cls_name in class_names.items():
        data = df[df['class'] == cls_name][feature]
        # Log scale for better visualization
        data_log = np.log1p(data)
        ax.hist(data_log, bins=30, alpha=0.5, label=cls_name, density=True)
    ax.set_title(feature, fontsize=10)
    ax.set_xlabel('log(1 + value)')

# Remove empty subplots
for i in range(len(UGRANSOME_FEATURES), len(axes)):
    axes[i].set_visible(False)

axes[0].legend(fontsize=8)
plt.suptitle('Feature Distributions by Class (Log Scale)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Class Distribution & Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
class_counts = y.value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c', '#3498db']
bars = axes[0].bar(
    [class_names[i] for i in class_counts.index],
    class_counts.values,
    color=colors
)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}', ha='center', va='bottom')

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=[class_names[i] for i in class_counts.index],
    autopct='%1.1f%%',
    colors=colors,
    explode=(0, 0.05, 0)
)
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix
corr = X.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Highly correlated features
print("Highly correlated feature pairs (|r| > 0.7):")
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            high_corr.append({
                'Feature 1': corr.columns[i],
                'Feature 2': corr.columns[j],
                'Correlation': corr.iloc[i, j]
            })

if high_corr:
    pd.DataFrame(high_corr).sort_values('Correlation', ascending=False)
else:
    print("  None found (features are relatively independent)")

## 6. Dimensionality Reduction

Visualize data in 2D using PCA and t-SNE - critical for understanding quantum feature encoding.

In [ ]:
# Prepare data
loader = DataLoader()
data = loader.prepare_data(X, y, binary=False)  # Keep 3 classes
X_scaled = np.vstack([data['X_train'], data['X_val'], data['X_test']])
y_all = np.concatenate([data['y_train'], data['y_val'], data['y_test']])

# Subsample for t-SNE (computational efficiency)
n_samples = 2000
idx = np.random.choice(len(X_scaled), n_samples, replace=False)
X_sub = X_scaled[idx]
y_sub = y_all[idx]

print(f"Using {n_samples} samples for visualization")

In [ ]:
# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sub)

print(f"PCA Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sub)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#2ecc71', '#e74c3c', '#3498db']

# PCA plot
for cls_code, cls_name in class_names.items():
    mask = y_sub == cls_code
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=colors[cls_code], label=cls_name, alpha=0.6, s=20
    )
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('PCA Projection')
axes[0].legend()

# t-SNE plot
for cls_code, cls_name in class_names.items():
    mask = y_sub == cls_code
    axes[1].scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=colors[cls_code], label=cls_name, alpha=0.6, s=20
    )
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title('t-SNE Projection')
axes[1].legend()

plt.suptitle('Data Visualization (2D Projections)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Quantum-Ready Features

Reduce to 4 dimensions for quantum circuits (NISQ device limitation):

In [ ]:
from src.preprocessing.feature_engineering import FeatureEngineer

# Reduce to 4 qubits
engineer = FeatureEngineer(n_qubits=4, reduction_method='pca')
X_quantum = engineer.fit_transform(X_scaled, y_all)

print(f"Original shape: {X_scaled.shape}")
print(f"Quantum-ready shape: {X_quantum.shape}")
print(f"Feature range: [{X_quantum.min():.3f}, {X_quantum.max():.3f}]")
print(f"  (mapped to [0, π] for angle encoding)")

In [ ]:
# Visualize quantum features
X_quantum_sub = X_quantum[idx]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
feature_pairs = [(0, 1), (2, 3), (0, 2), (1, 3)]

for ax, (f1, f2) in zip(axes.flatten(), feature_pairs):
    for cls_code, cls_name in class_names.items():
        mask = y_sub == cls_code
        ax.scatter(
            X_quantum_sub[mask, f1], X_quantum_sub[mask, f2],
            c=colors[cls_code], label=cls_name, alpha=0.5, s=15
        )
    ax.set_xlabel(f'Qubit {f1+1}')
    ax.set_ylabel(f'Qubit {f2+1}')
    ax.legend(fontsize=8)

plt.suptitle('Quantum-Ready Features (4 qubits)', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

Key findings:

1. **Data Characteristics**:
   - 14 network traffic features
   - 3 classes: Normal, Anomaly (zero-day), Signature (known attacks)
   - Moderate class imbalance

2. **Feature Properties**:
   - Wide value ranges (log-scale needed for visualization)
   - Some correlated features (can be reduced)
   - Clear separation in t-SNE space

3. **Quantum Readiness**:
   - Reduced to 4 features for NISQ devices
   - Scaled to [0, π] for angle encoding
   - Classes remain separable after reduction

**Next**: Classical baselines in `02_classical_baseline.ipynb`